# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq1PWtz27aW3/UrsOx0SrYyKzmPptqqs0nstJl1Em+Stnev6+FAIigx5qt8yFYy2b91v99ftuccACT4kp3c3Exr8QEcAAfnjQPQsqwXaZnmCxbAj1047OhnFvMo4uzeCbN/rZJ9xZN7J0fH7Ht2HO/ofSluyirnePkkEokvcncy+a1ImS9YLtZVXqTFYsLYt+xt6qcFi+D/OPUF/v5VCZ8nbM3zDcd3ImHQ1i/nvzGOFzvxntmZyIMwYi8e/+35i1fse4DE2OOzt6+cKeNFyJKUbfmeiQLKcYRQ5oJVCachsD2CKcJNFcIL4VI3zqChTECvWCFYBl3E1hPV9uvHL5i95uutwP5nfBMmABUuz8KkunFYJPYwxpR6gSPg+Xob7mTXM57zCMa1YApoDh1RY2Ulz33oithUUL2QPTmNWJnzFX8Hr0WcpVCd7TiC+t4Xu++LbcxsPyzWKT6CnjlqABJF1Pu0FBvB1mmCTa154gMe/1NPBLOfQrnvqCqUyQEzIqGu665STRNFbCMSQOQ6TJNmNhq85aKoopLmCprfVDgoKogYl2iTXWUwGpiYYou4i/RsNGh7VxWlKBBTmSjDHIvEYRGrgqJgYVKUPIH/ROpOLMuahIigEjsMNJVDX92gAsKDojBB60C/36zrqyhd6WvsSRTWt2Gqr94VaaKv00Jf5UJfFduqDKP6rloByteiqEuW21xwP0w29YMwriu/DzMgXTHR90kVZ3vsbpJNgjyN2fnzM6bePY/5Rkzlz6usmMgCLmuGHYSbyeTX09enbAlddTNebl0/zBMeC1vf81WBv7bnYcOe5zgTmD/v7NXT/4ZadWfds3R9ZTuT35+f/vEGXtgWtJaU1pRZK76+wt9IBHSfh5ttaTmTyeQr9gaoNw+BxBbQLY6UUrBNCMwDU8lSFvCowMGlLAO+Bjooq3gFTCAiIJZ8F/ppPqHxucDM3vMXj3859c6f/+30DLvw8L43m83w/wm+ffP8BMd5PLv/CJr2RcAKHgiPmrXDeOMsiJaAMH7nEXQJOD1Jc5BV4XtOxCg7iDOGb+2SxzxJp8iN8DfAomXquEhYCCcMoDrQRSHJbk1tqNlw6a9qEP/lPATih3YrcZrnaW5bwMhKEODoARlGD3bUP8AgVr2esi0MC4C7Rfhe6Lav2U/s3jFDlqGrQ22d1ZAFCqYYhCDwI0g0EKbQqh2HSRinCC67cVSz1MS3APxnNjwBn9HgJgdpI5oGYn5j4/AcaETPYAMWRqzGvU6zve2YL9xyC4SS8DCybV1zWsNw9DScPX759O+v3tQNYk0UrnLmEiDi1788QZKFn8eWM9x2shN5aasiVCIXIEUSfK0IbR0JnlSZl1ZlVpWFfSVEtnw0A9oJEy/IhfA2q+V81lDgE5CsnMSaIR4BTWwXincoKUNSUTXqFK+UnJFwr4mwzPdNp/0C+lwA6wvf9pFgmY+DRKHm4p+a5d+lYWJL8eC++u2td/L8NeDgW8txEEm6VFiAqLB9p0E8jgSbIAEHgqS48qoCuasNy3GpIOj7b7+9N6trX29BvmAngQiYDerG9guceUQW0jFV+slEmTEhND5o24fOpZk9c1pvVI/yGDS5sP0pCzfA2sITSI7F8m1eiXYFY5jiJixgynxQepYL4tfqtIr/oGwu4nQnzGKtUp+PGnGzFlnJTukHdGjTesZBaUwm6wh+2WkCdoWQL5HkPA+YtvQ8uxBRMEW1HixfpgnogyjdLLM8TEpjIFjIzaCDWA6Rrbrli1KsS4O3qCBAgKLwt/0YULDCF9hM+02x5ZkYf+PFuwMvy4MVR15nHAY4/sa7ChMkF8Um9ct36QqZ5IPF1yXaYNaCAZdaIvHWacTx7mO7ggfWHvDkNVR6BrqqaesrUNtogAHbCj9EDrXxAmySdBdGDplLIP1At4M6W4MNXCoObsMXwNLYH1Iv0P4jF/ujDGS4v/eAHqykdQYPHs7cXh8BiIe0bGj5YRZXXXbRhDEIuCVFzK65VebzUthYHqiC+3aaAd+2W3UMCTFOzDVBt9qQ9h8g4GOX1Li/7+GcXilUeFmaRlBiHbhvyUY5h/vTG/AewBuxQbN412l+JYD7j0GNScq4sNQz67LpMqAj5lcwdXnRQxjJBi+9UhKkYT6RFGBHEiIMLgOp/CZE22Wd+uEmRXsX6KxCC0OZzYYjs+e1EzOFl3EWIS3AmxocL/NwBcMB+6gS6DAQTSVo+YPBjZ4QaA+QAuuQkzVciAJQXusGLTLlbHl+uAZx0bxxC1HCWDhoH9tSnAiqEBnKOVgKefKWgg0PonK1xoohN0KBQ9w4VlXzJVQnGhkrBzilFj6Nv0Zb1TQPMD+F08bgEflj/4wWQTX57kbAW5PSLVDNBeG8oxMv2sUu/xWWmEjBdvQv/JM0XTMKzBKKDe9aoE9QKGUFHlxaLG2YBjCbk/J7I0pg0ez0Hsc7QGKL0c5AjZZ1dKD2qdEDJ0edeMUX73ge1VECcDJb3EH+0rbagGbdBHwtvG210t5TAZ6xR/MDnJM3VcCGQQ1ycTkuPdHyytHyasOwHZcG3jcvYMpzeueFZLGNFKthix2VcpEFkOVHyjb9/Q46XJS5Hbjk45HUJlCBbG5Hz4vLUTmOHqgY0BFgI9jWS5J0WQUWfiQEiaI6HPLrswVMacf6UvbzbNKwBdBK7tvUsXYzCSB71npyHYKmIz2ExdHfXEETqyoIBFg9m+XMwd4GfZxIAxRF+TC+VtCUVD32w/vsp5/AlXQGCyrXbzWOdjXCZLBAgvOB9u/KaTCAxgwGAlz8YxhkNNphjp4/pJGKm3Y/yrTkqBmLKrbFDSi3zM7BJKYZdvqWnh1Y55JjVAhKRdWAWRbsgwSm7NWFOw8+sl+e4GurP7TA+mCMgB2xcuHOgo+FIf8VXgiqoU6lYsnCTBgSYoqe+JRBhVVqEIVk2v09H9xLaSWin6m4thYZJ+HbZ1F6/YKXIBiSzTmAjkLDVMS2AEe3lXexNdQ1JTi5CfhVHTpWvQzSCCTwkvpbFcLDwANIsaLvfADp0IDaU/YVewZG/vb3E0C5L9B6CMI1xTNlKPX/5rMbcg9znoUqlDgqfvTwXPDtQTd4AYLe+bEdrz0ebdKlFa873oti97f7THrwfYCDrYy21GebOwiUvmBhNVrQ6gGnKkuTEBoaECl3hP/JsBXB4igbeiVLWBLqNRj2YBHRLKs7Mn7MBzTh5oO/0CQs9+SvtbXaU2RA00x02VuszewHaD3D8+/qfjsYXAFjE+NTFDuFroEzZ5AWBVHB7mEFjyjKWseS2VNAEUaEZUiGlTxehVCAouvaLgXgNSgVkpYBM+jNTHZH+zmqF08pcOW3lKwOcqagohtph3cuBg7BdincdeVzEFRlXEUuEHx67ZXBvWPgT0TagUpJ8qnFV2BbbMElu+qWbokV6emqnj8BCJs8BcS9phhAbjKzOZ/DFlqrxLI2u6DfXpp4m6wCmw0mDxVKQy236wVo23DKB1tuOe29UbRhNU1TeKblno8DJ04imgWqYuZij6JaSX4ktbjjum5H7gxKlU5oQLovhoYYNhtHVfHYP2urqvtHflge7Y6PZkeEABAA/YDRZ8gXiYLbBNcAibTosktj7dn5rLmRswJOp+TWgXlpBXXuOAFDCO17Z3o48Y5GMk52FPn5jNEBYEV7O56HFFEtSl58BvGp2NOdhg/+yadT4CARxrsvS4UNQg5RYmda7BbJgXQamRinP2ufOWft2TpEm7dP27930oamrE/n/8JsteZpiq5VVaCSLVJQwHNa7RyZPRkUradE3g7OR0vbleJmyIQ+x+o9w5l6XFEU1b4wcEGNmVL0sqNWBOAH3GYHLJjhetZlz4su0B+nFbniAN6CIaKi3jP7wzfU+jfYmfqSIMoOfSMtmm8+DlDaQbt39EUvRD2M0MOeRY/ohkVuy/UY9lVvN+0NKrLk1CkMLQ52iVb9vigGBgfpTA6D15F+6O+wUw9u89WXckfuoswPL1GYplvNn2j79Xm0ie8XApiIQpbgWobvpZOMRiQ8NEPZvQYurDQI0FuxLhe3dFT7cGToe+us8lTVjjOHXHOL3aOye54/buOL2WuQYDkuQe3BT0EHo0n0cKyWOEOMDFnK/UBbRuu4Lfk/7aor84EzGH7LzGkgCVo/Mc3RERfYLVPbApRZzvAqRsugozfXPI+rzB6MxrxQCwQRqEIZirk1rKJre7FA57t2VFUz/WWK3xKODhz3MYcmCcFGB69OJiAtaDEijKQbeiXyBKYcZ+rpbyeP2R4IINw0o6Hga7RJ87CMZfh1XZ28fCkTrQBGlkOvc53MwzFXiWPcFjO1onR9N29RBWtajkHLKDFJhworpacdK+kKkpcZFh7fcRgeUKTdIQbp7N/qevUDvkOJOSc5v26zfQzwZF5CIq5VQsEUfOn58ZTBHweuZ1Om/nOc3tp3DdbFP3YYd4q4IorCrBC2Pf/h4ZTdBzD37sHF8WzmUPwvWtpwPWXzH/DPA/hz/OBBtyGgWmDOZBMhnB+h+vxHBDRHiD80gB7CQ8xwmD8cBIMup8T6L5QdhvFKX+zCtVhaOA+WA/5+AgYmCDcQ/Z1AKwU9ZfUwCUSOeoEkkw3ty+e8KtM1L3ARBcFNmV+CilvKlyi4yvlDZzgwZlNuyjKMp5he5TUtFKXIiiXMRrrGRAIvF2BzVagflvPjRyO2IoJYgz1zVSy1xJK31uVUpcZBp5YbgErZIR710yqRMYpt17htCLXYJ+ttDqLzvbCdyYjhgyEcMJVBaSIr30VUfFJkn2ngMWb6pSyNwzL0tflpxG+HFBRIXbg0k2qCW+3SLgcOclrbbnX9MKAJLENSYDmthoncRfSqO82YL+DRa3oyGdGCaiHPHAxKHp14NV5DNlRXUJQgn3YsW6Oy6t7S6JmtVgdN6uuC+yS3xRzLkqak3xucve44P3cNaI2xRJnDJ1PoUCuoFdcu5Wid1dFQvXlX85cVlFo58KatMPDfDsWtrODuwhyX4bGpNN/bY6JKhiGpjIdxxTXHVCpHL39Q8H5Mk0h3YjbGqJjfWpG+/bCL3aoQfmtR5Xt82llrmcm1ln/+Y2itRa63IPt/2CgIiki+2eU89jarby6xuvVJuRlfYgFYp3maa8CZEgiUISkzqbzVRi6NtOQD5t0dEAxtWxrLd7IvizJ3upSzqdUuLRpiJmgnz6+TJzoZqPsqKzBTLPDAWUnAoi06RbccqCHKtlylDVKW4XKpMgvJokwyl+c531PFC3A3QateumB+AfH8BCp0Zg6sRpKsmpZNC0Pja0K9djdp0cw+OJzWKCcf1D8wFPAv0GK6eicoJR7dshx5WiaNV9zP0X7HnJYYrEbRiGxEwOBI6xL7YspuClksSZP3Ik9tzn5m81nLBcAF0puiM5s3YGzczKdsD7/7OcC4KSQGp3TFb/BqXz/bq2cDGCMk5Glm2whzT3DZd2xOcOHXwNpQBi6xNz5MSrtJY/2Wzd35I8Mu/2vY5AMMFKPmXvEXKAcwRSRF23YBevwahBAIBqwKd1t55/SWqaAqaA1UwPZ8dnwfrDP428uGJUVMCD++L8VW8VcjkossCkuv2ApRajWO12C+5BhokGnfU8z1nuLaylRle7fci5NwF/pyT8U2fYd5GCqwJ/c8gEchcDcArh3hajMuXgVp4qdtVyCQLR8UCEbiWL5Z8TYrjHFuJ3eU2uhlc6tRo0SRSDJkCL1znFE+azEB9qvmAkJ7XW5NKWzc5cneNIELsQEKgZ4AErEHPC8p5URFGaeGmkEXOJwySggRYIqitYlJp1FXDAZsV4sSCXnAEe60KJcsw47/r1AlwR0G1cvgM0fo8gxw6QOBY3PQjtPi/yHIQ9VQTuB4jdp/EFLJM7yYX7bQingsZGyRolz4CAkNygFjFRcznKE/gJFn7uz+ZVceYXES1gsSlUAsbFuBvCwWuA6KWxrCDdgZouilsGBtYiCX+Mu2ppbj9MaGvbNRpITQgz/AFEicKYkYeIAyqX4qs3dCyt4BX03YidN0Frd3EKjUBeoNM1uWTrF0tw9NLfBO2omYWAdBgdCRYtdBAO8BHjUwpR53yAyAXODby9oGRuWPHGAIWyXD6SkKK8C94/Qi6IrBAeIXMUtozavxWKBbPMyVgENPYYq7ETzMAivummxinwjl34lnUE+tq+JA8Mav1nBzR0P9GTqsIlcgGiRgz5CJWq9tx8bnvVKD3ekVHrWt91gOrNnWyxgJqX7lopvyRpQdjRoDU/q+h4Xsdllq20UZCZ5/ITHtSiT3QOC7MNl4yvTxKyBStMA9Xd0+UAVn0wNrwgOHI8SYuCdgzg5UWUegCbxtCsyKCpwu0EE6ns1mB9rx0hxdYUko3jrdkucZdX0K9D1izDmgbWcSM86g3yLDAO3KcjrVK/et/LVjwqO4gSHCK7J0YsKl8UBtNVt2eOkTnf56DofXonTPBgLgg6siB4aqhwsYBitybKYlZIkNOdVuEN54cumm6ND3aIUo0tPdKz+In+PDYe6czBfa9SjDItK5Pe4ZsQbNs58bKbMYmnNDdCjGbeQSaPdlI6O6cpJQ2xhxFCVF6aVEHOZKo8jGCJeOb+F8Cn/KNhV0HkwelWvXc8uGPOy4isoQ9cXVAiwBVBVXygqhltywFDFMo7Q7jNh6o170erPUjQQP8TNvjCOwL+Qy44FwPjy4rJPq1Brj4cWAS4f92TWNdF/UGmWnC5Q9a8SeTZAHGm+1XF52o9mJuCntrFnDkG111yKaeca8QkKXDHDfuuhh0GFBiQxD9cpD1QKq2RETSEAo3MDBQsOE7kHwPHKkzaafsJ+WYOhKnDz4vJWdtNyKvNPhNk47ne+PZXCtR8HtrPe0niKqhtd6qNjAek+dCkmvqnr35kBgV8RZuZdp4XbXNZdsBZ1Us034I466UPttGzr6xBA7zQ1cOJ3c4i8eZleirBVp3wyH2g0KGjWU+rF4rCWfOtOD4fdRmFrieQXGt5c0Dls/dNph+/GODYbzwYMYEs1TNaNfxJZVwdRa2qsIb8uaVQIf7YXPDcMrSG0PLgxqmOw/loCpkpelNqQtjCPrvUED9kV/oaDun8kI5Ly0qf4S00bko6tLqW/kBt6RzeekeEgT4VaaK+fAdo2v2O8y9ygXkjrx2g5yOtvgOzwmgkfAtq7rMLmmCVYK7rpgUcq4D6pfdF2g2vnhFB4iLNacUGi7AK9J40kux1sgns822vShGhiTayVToRFH/QTynsosIjm4Afvt1s63eti4aDJZ7csQd6oS3JGyr8S+bcCkWWn6Zyho1EENKPfnhkCtaYROLGjjDdWoSR39UMnBrIyt3hd55QK5kLx0xgvtQCes9iVatReLBfvxBwOD29YOS7+Ks8LGIU5pH7cHw1cbCAYaUtjfultx44fg7JTYwPz4sjED8ypR6JNRO4bcorcJA8OoS+IZdU2hLXndD5N5B2OA3dD+tG85qEak9FY3ZM+qayBi3Q9QVcv58b37bShaSC9p3x60u7S2FS4yhT5upNvSPrPl3H0EL1MQTWI5g4LtyYei6dJ6ysGG4bk1lQ9QweimOW7GrLsESC5xjWdgTEqUqdz+XESh2GGLD01TmryyTY5uWcTjlc8ZCukFU6hIPAwdhTwaQDr1THlzVBT4eSO6+wbUdly5JXU4jYYyUQZlRyvB57xm/knX4KtVNeZUnF9YdN3aR0tzqkqqG1lU3phlacpVUXktS9K1WVAuS2IxvJKFOklP9eypgs29LF7fm5XUFOtxqTs1MnnX6saskwrSGPNqam0gs2NQP/9ThSVlIlI82zK7CeJGbvUmjbYw4mOKOWvucfShKmYhYjSzTN8wkUrQrEQs2gYsedUsRE/MUh8HY/DD8tB0/aT5ba4fqBh6Izsc7RkOmuamTB62TFroRFMABPmkC6G2Gkasm955Jc/oaA195pAvlII0pg+FMh4/Ea4VGyxbrvRSe9RSmNFfKc3gfyXMagN82KSsJVvjjqN4g/9rsSZ/askmf6Yj+EEhR39NAVdfNVJO/Q6DaYTfKk0ju751prXsky/kjdMIQWlTqztnFDrISA15lTpTLfHombw2NN6VFk9oFpgGgUkBVzUlyk2o5Ny1jjRpXgIBXVjwCHh9zLB6rc9nwaUrqoRm09XghtYW4EZGdE+HcTox94M7yUnqFGUekOSxvv7fr+Ovfe/rX79+8fUbz8Js6qvhswzIKuscYNA5L4XO1Wm1jmdF4Q78oqz8MHW3ZRxZDmUuGZVRB2G3lTpCsWkrzSbXC/xFJ3aBe8y7Az2XW889Qv90bBRYZHQYtEIxFHZafKJBt3MLvhNtVMiWA4uMaO9D8tHNkg0u2tS1TzU5CuNUj6/YawwMkhiJ2Hw2Y1/LbXfoLuCRDXiqGybJoB3AC6EPrcMTsfIwzY3ta0gbpxfq6AJ0gOzTi/rogkvliSHbqdwTJIbT+jwAqnG/wZWn9/Pgyha8mrk/zvBAuaaB77FNo4JUvrpiXePWXkg4rb3dZbi+ssH7WKEsXONCQEHZGmnWmSvLsh7vePKepPEKF1BZUcHcxOSN+ZiGQAKajuuT6Ms3vLVuTMibj+RuNvvAyTaCDmAeT4FLGn2yuFGBLhj4gymzO6l1cxwprv3LEc0HfIDaOuA4eyuoxXEN74bG3y9O3bnmYQktPnDaGMzAzRF9FAZJV3IBjNa5bac7QFoHA3RUTFNE7i23S8zkKJc4V1MG18VyZMqAwzkYC8nAwUrAPbQua99hXUBJzSDpFA7ALo+ioV0+iJ+it/yk2iWuNbBWH/hzUZ8ocomx3LmpLfRRdyDs11eY7NXJijLMu3sgGE8TXLPnC33mJPAsehEsRbKUe2StTnxNtzCgYLqdOzI71ymlj0jpDGEUuclOHQHQEn0kMg15uUMjqBhyX8ncGeMgHeajqJavwtJEoIAnYBUlM6aGdLnrSqglg5loQtfbl++dWMg/VndPn0U+zq6JoVgj20X6/wjcP/8ht8ZZnSA+gLlrd6XeW5iRZ1p1ObDe8jn7BAfsRH0ywsDU1aK+I7GKcQFVz5h1FsZZSPjHtK9cTgXfVO9EnhbWMB3UO+7Usrqkjc5CVWu9T9zgotKQxrV4vg6jiLubaGUNjG7Udhmq3jFdWiaMbdHhCpYsY8YDQZ9nnM0XzQEMezxdCfT0p7NJox9H9pHJlz704kppXMlJpt6dzadSHxNH1cr3E0nJOg+TUnOWGS58/vhTN0K2yF6Hn7uB54HJG949NIKHofBzQ+OyzK0UruEeojgpuFVgOnwPxv7nEt8YpMN0qCe0TYmaDo8XNEm0eeru+xNH86LvvCJ1t7zhVv7w3bT4kFJrqT7DJkVz9UHbTP1OPVJibtLnNOnzERm1m29Zrgbo5mEDHJ5NDKten48cVIIOlor00cno+5AtADJCrhu1T0em3W2uAemEztrBPA3cO6Xs/wVYtKB9iEZ5fZzVFJPoybaQmf94Uq30ihuA5WrshIgk6/pct1Cp0cnz5jwRuWQgj/jAM0BC9FTkzC0oBU72iCWh6iVtFtuvzTQ4InRa1rHkGHGphupJF0KBkEvousjl6FGMZnigibAVGA6RITY8O44OdgOjFa4tbM4H+wevZbP6CFjZsybiJXpQPrYNcYE601YmfJu8ZF2gVKIudA2MJ1N240xGdC/KMrBIwM/4AaQY3aOf0KpNHkOrJ7vxnux6Pdkd6Il+h8ii8sXGJP0z6bgCfZMfLlAVggQMOTmtxQJpQ0UM5DHOjK/yMG9IxoAlEn0YpjadI67JHnNp3C8To6jPNrNb9sAB2gf8l1UWif4+7G4IQIYd6pWlrv+Vr4e5LrjtBFqoOeCDmmiAEt1x1puNzFMA3aJaxeBF9n0JVQqTSrOpXLgzAow6sqjCcEbgsA4XdtY/dDywjvvhxGgGJ05pV9B2g600nyiAsiJRpkr3meshUorXtxjQMpN8g96ZsriBXTF3zdjdpV48kV4Jm5evKMzL6WhA+gjABi0aENwiMQ5LYlmEe5j2rGOFfAVGMVZJcKPE35+fE137NV0blFz39m4T9e+arNEJkzibSnnSPdhzJQ8DDoQrT6o298F6q6hvgq1MNd7EogxtawaotLZdReML9PXaqH06ZQMH0MIQri3n0zc6SSUZpO2057CnNQnzuG6C+Stp9/DcPiOH3aCWaqRzii4Uc1r90GvBlAREMmep3GsdbedeyJfSsJnWh3l5ZBgt6Wwnu9R+3HzEha1rKYh1PYIJtZoSap7qEqvoLmBxE5uu0qaMmYTfj/oPQlTfe1jK5XD5SY/l+YUlr2hzbVbhA0ofazDZUAviVBFMmEkiQV7CICJu8pULph4v1mGol1jV3ssl7vow1n4web9PEQo/uE6AkBG+e52HMIPINM7QGuGPjzCIFGccxJ50xUByGMT0noivKutTxjsmEGbfU+PvswH4czJ5zvDcgNaZi/XKlR/mS1k9zJbvsymR3VLi6Ursl1daisjvZCyDnX5AFsJSGgodMwQM/bDYelKq2lCzQ/2DLspOiZWq7MuVhhN3h1lRttllSIOlbmG6Qyy8c0b8qt0Q0J0zclJoErQO/NWGvTzzVzN8uJMH/rbf3hEiCAqqcXHpQO9L3Boz6stzj4wX+AEFWbdqQEBUwFtlG2NbvSKj3bqwaiEgaYGkvm5FvxuJ0DUn7CMye0VaTP0leNpgjgsLaUl2Ngn6h7PVx83eQQRwSx4+ux05vXerBIT1Z/JngkHNXUtS3IHZP+tgKTkfA/leo066dgnIabrsnmHSMU7VdxeknTv06QVTWQYH1uKCHW0kwUa9NXQMcx7slnwxtHwvlHTX3rTwo5drYYi5KMZOVOksWWBJevZfiKZwHYtym/rN7iJz5oxd7TFKdxDxKNzhxmovFKhP/7h/D7Nn8GtDCSLwafPm+bl3cvrs7PHb0xN5ZjyK/QisuGg5J8J730/OIDdIfRUEKBhPlcFFcLKm+/OOQLGTRotv3r56fXpCU+fiSZrYWdu23Di7jw4V+UT4qxwsabbit0T+HD6QCn38oRH1Sr9XzNLnuwCPIWkwIHNw8a61iJwLMN3XCpHIQyo98dewpKNqIjx+AIMVOrZz9DPjO4ytoyvuAkbxOBtcqFXhGqr/4vnZKaDk5Sl+guhDs+tVuT0LdtEElmkxxFLgF7QTntdrG7LQsPXT9oxcPCGmAUOWEUEhx1uXG4MEnb/Z1wejE7hHJjhgtHUVETxZYhQQRaxyvW0XId1rj48KyHOLRqHEPJOfR1MOCILBo2oaMPrFKGpkuMDXK24exxOMfMLUA+zQqSyAI/rl7AnTr+8M71qsFDW7eLqNhUDw1KM8lcfdwvu7AwNjxQ1WNwSs07lnT/52BzhZKN5zwvYPSAbndCs/llYBwu8CQU/8DzjxLzCoXJPDnccRpfgxB/fRQwDxu8yAFoVEC+ZLn706ce4MTMkI8AfIYn2sPoTVsKLlGIsbykEd4Kzf8UWLr0SxBpd+uCODFHxswKl5SdLwGBQV5SSywpgVMQKO47UO0mJPqNM4pKbcYXibEOjr3sMZTVQPXP12BIjsscbo7zLEoVlRIfNjkxmsDBd9LsfnBhiMgMKAaWWkw8rEVdzxY8kYj06JrbNgKUW1m9q6jvHgqwuVFPXk7PTlySl+XuOIVM3RUcDXZZrvjyjnoMroIcWWZ3R53l/XGgol6pAMsmq2J0Pu6Eg2EOLsEnrgBjAk41Q9qPASKlsKf3AnUWjR+QG2vJFgJU7VC3njDIGrgzIS9/oJJbLKyvW9BKzTVuVLdTcIup4zHOJcL37LR2oBXKLvqBHR1NuxZD5sHefXUvNMdZsPtKg2ZLLYfyxrVVmf4VZH9nXjTeSf9h9ToUbvkkOhGuq8atYOGksAz5rsfLIjy+WJkWq7rHtO9j0QGw7Ahwlemi+fn5/SczAizedv3p68+u3tbQunGGZRaeCrKqAtxnPjczCYk7HsnOeAHUajLctd2ZteOpv1mM5AYEHOY0HRTqqj6soMHGmnWc/w0KUBQy8mgxdcSXQ67bwFkdl/+t85MHMIbdBdjuvgrOJjeeqszAxc1hJ72NVrauGXB21KrnLvgVVMG05cPKU8s+cOplfdp6PrBgTrgBsJQqIMk46XQpOvj4wg5OTqTIR+OF7JIHkC8rhVYKJ7XN3rUkNfk1MN2e2kg2YRZc9vWUfpQho5ZbRHDBcopJzu3PV7qE4Vsa/Efqqz1donjMTFWHwE/A3281JSNjYEMDQuxg+UVXxAp1yMb7gzCKebQge8QolzBtunm/IGUyjB17akuCdiaEVuM/yIDgo/3FyDsmk2kgcsoV0c3ZvNZov+Fij5ejKZeKcvf3n+8lRPiPw8JMgrT9D3++zux/nwu4w8Yqqa/k6lhtJLX2/Anzbw6hCC6owG9v+FsGBO", "blender_rig.py": "...", "app.py": "eNrVfduOG8mS2Ht/RbqEsyz2FKtJdrekpkQtWpeRZIzOtCXNOYY5jdpkVZIsdbGqVJe+iGjAL17AgI0D+GWBhQ3vAob9sFhgDT/YwL7N/Mn8gP0JjojMrBuLbPZc1rAG012XrMjIyIjIiMjIaMMw3oaZSGb8C7sSU2a+TrjnR13miYCJNMvhZgQ3YZZE9CyMMjGNogsWMTdaRizmcz/ke3E+DXyXw7MQ/88SnoqQM87g6lKk8CnLQ5blIYCA6xdBlHuzgCfCNgxjz1/GUZKxeRBN9fWnNAr1dZTqq4SHXrQs7oS+SoWbiKxolubTOIlckRZPskUiYGDhvHjgL8Ve0TENmvEUrvb2Zkm0ZDZT72A0M3+uHooQRiv0q7nIHPlkb+/092/ffWBjNjFe8CVQJDEsZryIkkTQ1Qce5B7Xl5m8es79QF69F3GURsb53tn7Vx9efURAqz0G/x4AwZfTRIyYGfM0Si0WuVkihMVcDjS2WCauszzhXZsN+oyawMwsRQi/x2wZwZRF7GOeTCMCZ9ClAdCOLXb4+MhiD/vwz2LD/tHjriXbvBQZDwLuUbvHFjseDC02OF5v+J7Hvmw1BHDD44cWO1pvdQrDxTZHfdXl4/U27/i1v6RWx/1Gh0f9k4fQ7Hbv1b88fXf2zasPzsu372FsxgHyGTDmwZs8vMl5ePiyNzzgaQp8cCCu+TIOhANA5yI19vb2PDFjzswPRGp63RH1GiWeSGjKJJ0cHkJ7L7LnwRQnRT0FsWg8Sfy5PZtel0/saSBCr9JCzor/pYBGHRb/dDMQMJ75bmRH00+Vr4vHyyyoPAauS0TqR+GhZ6f4pgE09sUXnmpcYwG8oG944vowpfoWBuD44SyyUcqA6/DzGTLMBD6JebawP0V+aHoWm3XZLErYjPmhopc/Y7qRuPbTLDXXv+kWIL8CmLMSRApiIzwTBd3GH2vfGvtGt9utDgz6m6HWwc8BICgAupuBLHrplZ8tTNOwg2huWLpb0AR5go31tKf5csmTGxOHrOaew2CJAiDDpsE/5WkGfGKx1a3sPAChJoIUqMyM/f1V8UmHxK9jsc6fd7oj61aK4/4+++F/apFkKy6bwn2ne8via3xpVACK9HMuApFFrAIY5gZbm5VHixznEjrrwwt5YzGaXwZ6qAay8lWsPur0AF4X+668TMU8B1WaOlkEwq7GccvSClPNjK+jZMlHbNVZ5kHmX8Jk8w7ORwmm8qLLRJAK1hnAbGdR57aGltnW9QzBU6+E3UdFtreno/qQOqnf6FZR2PHLbkPssrUb1bjsqAr8PXCRSEa1CaBHmnA1gvz0p//OPvoCtH86YoQ+2zywkqa1wTCTdHC3OSR8WIwGxlJlpC2jKnppjux5QONo/XgaFGNs/3hldi59TwCDddhXLM0SswWKbNHpdqFJB2CYAgRcvmRxwMOIqA1jqg107fNiyEZN1ogxd2HZ/X01R1L6q50ZpNFdUJkg3EryC+m2eRwDGUzjtGwEnAeDQRUpNRJCmtSgnCvtVOsmiDyHFMDmTr759iU8mYskWuukpsaB9hfIyJeFUukapD8vLHaJOlCiVPZ4bvuZWKZmtw2xJQeTwElE4ItLsRm597LBiGF7tM9C5GBYK5mLZhNafik8DbhmSaPoy7gUCaxVInVSP3RQ50B3MWAGM2VofDf0OzPOUEeNwHTps98x2cMSbQ+W+vNcMI42qNaRRn25ayqXSWcLJp3zW6Zfs+JpVw0DyXulcVWMA0otwgVhct7diP1Pf/03//t//Ymtrm4VILXyGN+H34dqcumTLqxE79++JrPOOM0ztJkMjr+BCd7kS5AVfLKgKxALsh0TnxOp4YWbcy/JY+HRBx8A04Is+DoEuzYHELdqwbv0xZVInEW2DExv/HvgW4sthD9fZOOHaFLhe/X4yk/E+GsOEmgxMPtirp4v0zldqcGDkf5KOgPs8CUz0ZoWYDyA7IZo1/uzhC8FCH8CQl3hFuDQGKCS4wCmCywWaBChxa+4B1dyDwwLurjDrDCUO2LjuMBKKKdFkR1m5KnnX4K6ugnEuCNHPFrJ37fx9RPPT0Ev3Yxmgbh+wkEawx7JzsgV6AM9WecwAw0Df3bTU7ambokQeh7Qzs1ALYzcKMiX4ZM5j0eDPnQ05e7FPIlAXY0eDIaDw8HjNthTMqh66Hvk6WhwBB8CoCgZPTjhJx7nT2bQaW8G7kRwM0pvwEJZ9nK/86wFVHXc9FXqfxGjo4fxdefZ//nP/+3fPz2ABs+w1TPUPSZML1Ld+JBP0THjtGazGxbnQcpZWwdTDV6hKB6LEyE6z16LUCQ8QbZA3+/tKYB/pZnz6cH0mYHLg6EQoJ9a6tCjypMg8Kcw0UlauFWfc/AxqclnEJiZkY1BwDMTPTYbf5hdto8qA6yhgpGQpUuG+Ix258z4s8vxCl+UzZDdG82MPyMZGBRtSAzWGknhQOIReibdy4GkiUt4Hkg30gGv6AA9jfGqycKdKgvD4vXnq88KtwoLa2FK3HFnBT9vO5r0V76XLYDD+r97ss7ckptG/QantXHZOnt2QNUG0RUwTx4EKSh9ET5h10kvBfR9HvTAi3cvQMt0GsDlZ+VHMMMS/QaT1iUTF5Eeyd8oQfSfgHUOXnQvi+LREbLsU85Aw8yK8YO/DFp53AHDhYcXnQaDGnXOfDx9PJ3NnpRiMDhEmKfTxE9QX8U8JP8W4xfgI4J6+ukv/+rpAS+5c2/vAev9gn9sroQCrXNilBQgvvPTZSStRpA1kBDUkLCWsGjmu0BkXP8KVxbMyzyUJgeshYhnCri73BMh9wicKRfK3rPCTITLYlno2uwFNCTZ9sQlPLwElspLLQzCDl+r6M0DFkRsCY4FzDMuuqDiKIYB8LgcgM1w1YHXYbr00QdFSl76lxELfWiOgkveLsC39/75t89pqdPrkXPF/cxMM4uFwBrguiTRHBqm1cUlFtg1kQUWC+hoyj9FLAgE2QGIK+HB/gJB/AVbRogrWprQml/y0BXFunK1AOGjrrTfmGYTA+/BWiocSHzmAX2N81LWNWImjN7s2ycnFjWbgdVn4armjsE8gyedJRoU6FFVlVIPG3eyfud8ZPdnaPiXjiy1SwMhYoB7XFhPCF0kSRWHhPugCueJ/SpJogQUwtcoY+ChIHBoCz0Ti0rPNo5MZOOSkukiunL0OJwIXW6Y29THYB5HG4gmD1hNBvpw4rm3hGWwXJalFvZDmBM3k5GSCwy4NEEr846a2WCwhWivCBNQf55nWRTabuC7F11U7kB74Km0qupW6/BGbIKDAdJikOFCOgYlF+FsSaNGDfcB+06ND601WoOIJYldLsWXETKHK4I84GhhgH2C/OWhdRmypVgCz/IqSlW7CQ0g4xWxnJIGD4TgY4Qcl2Y//i3y/w9/d4hMoC0jFT764R9HQNWUTWFJRfHdYLcq3XUqIzMgjmdR8OPfzyOM3ClRDAt7uJxxZRxRmEVSIYa5WQ/BVMcVtwRt4i5Jgn4K6hV1JT5+xo7BHCfaowWoOgbrI3YIFxMW7hBkGVcRiwViBtekxkFYFkLoXw6tO/AK6HwpnOncIhhlBLNJlhkH45wCmtBSCBgFCrgjL+e576GQU1famoUbcAUzgRYt+JJrEOkpOAEANaV4IIIHgmQcHGAdOkWQ0v0BFiyuwQLmF0L5qGuANdOOgdfPtM7oVtQZ8c0A+ICHX3hVod2U2rjwdYCTwMAPYVGguRFVA5lozfyUpoJmjIirn2xUG9KyA1UuQ8GBsvBA7mcJmrDMjGhtWESfyOGjGE7atUvHriB+2QfeAbPJILyNv9Auw1Du/v7hAPTfQH4twjk0K6PjZrd1OCVg+XiMH9pgovuZQ6M01/mpK90z+kDh+slHpNQOgJ1FFyJ0FuLafKiss4y8rhlol77dx2gqukxn0gkMpUcF8wcPpedD+ri4oRVixKSHVGEDtZyAxjrHVn7lk6wP15V14ZY+wzVxAqieI65Z4XZGgYdaNADym9ikOxn1hv3KcoAP7TiKTWhpURegCvAFCiXyIYgjMGvFG1IrFob/+bWp7kAlBBHPoHEluorvlrJlCTMKHWCFuTDlWu01IONQ8Quv9lQtrto1xtsKmldRcmFW4GTJzagmUggiodVZ8kCShz9by2zTtqqzqhbSaoc8DHHd1bqnUDh3wkPHnriSNBP86G7STxWFdDeWWxTWnR8rJTaWc67uKA5Xqjqp4VEk7gRHsckxUoho12VPx+B/lXbcGC/uBiOV6XgaRbC8ltoVyKV5bqwvumvsofmuYBX1qGgorl0RZ+wV/UIbh4PFXGczvfUHzoxAvlrrhEwx8uVW2U0MTl7XdhxkZse5BQtM3BprnxAw+qgAa0sbH5Zq16wPJE4kEYvPytczsDCCoEUupI0K8D8muZAyVWxm2h/pylT+EcoZSCwHWQjH2LxrAzWTTGFRmuGGNCoqlriUVUXfkrDaLQXFZdXDSiqKZLgBvzFUHAm71GEkvQYekIn0ThoxdR6BxSqPPZAPMwWfxc2EN1aINRpiOHgdHClNvr/PzO1WOLhCZ0B2chbKCOZP//q/NjbQaGpA9rvdqsmjmksK7LbuD0flIg9Lv9qExeiIAlYs8LQ2kYZH/QFdVByDu5f4j8qmAMsWXEPadPeiJ0z1zLUXqtf0yvTrKO7uDNA69wDGaMz3UE7QRx0mvnvGs/am5ZzXQd5j1htRgrMo9AV5jEVI+YZVQvvIEbWpJ5G8z8QfjkrYVhU0dpS4C/BN0v/nc+/guBxnbe6TqnKV3GBRQBwtPsUSFl7hM9XikkyvEGbGNLftHKttX2iCswegTK++yVvs9EqFJfd1DblcGtJTMeZ+Eh0+7BvFZjA9rjlEXW0hIbgv6Bh5GHy0v/ixJDolAMBjc/IlPm/xir7Eajtqco5hyyJhYLMsQJdtXK0CfU2urvqwVv1ur7FUWtWhKVIoEtjL+Aggy9E0xean//hv2P7+O6l0kNzIIXcEKvb3accCR1zbK6/DhlXFzyI18EJLTnnmLkyJinTOlLGzVWZU7J8+u8OJueTgpqfkv+iIygYHA6YZcAgifIl2+eS8sLN9S3KiCPMlCAbMFXXdbYn9+OwA7M1QNSjiPl9HtE8PNBrcHqzKBreVKA+yEpoPtAmX+iEwM5iQ5gxd30Rx1sxGg2KzLZxUbGCER0QdtxmZROcx/awLE9JBm+IgsMj95/UWQKPKNhZwTBGqnlLiFrAJ3nVv2U9/+R9aXiaTDkDtnNdGv4MFVu/3P/27jf1Ka6smd3J2a3tr0Vxz4fLGoXU2NUul/JoHAlgH3Vu5Bqdg1fPEw60/lGIMiMWJuIQmUcM97pY+OG4QET8VvOTdnckik8bsb7/7iNlKKq0FvRQMwElDqcJ6Shg0BD+VGrJOOlTyfpiXnOMv5zspX9SjMFRHDtWOw7lBCgWfkZNFj7qt5vtOyT41rau3oJfzhu2NdNRzb8Jri61NvQfe7+D43AbfPADz2DRwlTJYkRGkGIFATUYP++dq8iMAKqffFJfZCBXIB9LEL3nGFRmBpDSN3q85hXsbaKVnkFYoHt6YdxOxmK873Sj9b8d5VUlRuFAjDSZAIdsPPXF9vtfCe/fee9UZl4s884MyJkC3thvFN3VY+gZQoSl3aIl1nO5aHxh6kBhKG2Ql4yg1faleYRabDa4uWBjACevI19PdNDc11VVlKeBp+v+nffPbmii7mSU1aPc1tKymBbLB6ligYQONFCd+Qdm+rzwT2dplWi85oGaOCzUD1kGUJ64o1pg6L0pBiNOaIOiQQwTmf/HwcgnYyob2pZ9kOQ8c2o64qcQLULnTZ7abe9yG9w4aPEgVk0I5lZd+6vBLTCaeBmjfEWExLtuvLM5tBlN953mlSIQGT+JPhSnsuHuL6+3703ewIF8u7TwVHphHGO/tj+wB2I4HLfsbAAraygSuojEamq+fY2bT0o5FgokUt7+jDLHXZ9+xPOUeH7XDMueTwTkYrfNJH5il0fuKXjZ7qaWzlaCK3c+Ugu7cXQgYF9pywqa7wp5ptWWu11JOMDNOXCNTOm/efvhIkeZMR4Xnca4vLxO+1NeVS7dsgXuodF1sd8kIoOa1dfZCBrlEJkE9EF/Bj77dX+dLVJBlPjyZlUYIcuvzXrr0Wa8HDmty0wNkxwja/8JxvDbcW5InadL1NU2qtX1LC4eCNnaeCAITR6CLbC/hV9CbjI2N3fTSCqOF4Bi9DaMcZDw1toYQ04UIAhVicnmM0J0oz+I8Uw8xkKAvwc+Bd+NDDIB5cFUAxrzfiQyLXksFfk2pzdAu8WOzK7cfTLAyjS6wm761jG4ZaUQXB/qILUl1Du0sSmHbh2tiRgy888kQH4N64ZPDc/x5VIJAfnCy60zGGtVXg/7wiNj6YIUfF08kTxt3LFoVkAbub4JUyU/u1DaXPEjJOnAzE9kAxwfAxpe4Awe/CnkFwsNrBQouHfUclwac9HEWlwl1F44laYvgdY5iiS7Jy+TCKfYMrkuVhOlixfvJqHfSrwWjEKBVjBeuljgT2hGoLRelL5DwGZ5P0VkL5kmfYkTvo08R7b55EePBHM+zpDJfLfBxVxzEZ8FvwPgOyqhdTbzakQFyFvK7t4NvpFPWa2kytWSwzrN3cmQY8sFENrgDbU9+kkpc2eYaf4qmqXzOM9RZIUyygQ8x593gboaxKdwiQzssdHC8eKey4b95+67KICePFIecDCWLnPQla5wcK0Z4fKg238ABk3n0poEMqbQi/HpwPD2ezvp4+Ts09kzjD7DKGFpXYouT6aPpbKZb7DWtX9Vctx7MTryH0wq8F7I/V/Un+rx/2N8I7SPqX62Hsf3RbHroneDlD//wQov/Hy32BkZDWZQnUtmihQEc4C7APa3HHdD9zOquo9xTQ9EAClssJ+ORaFTyAkYQCu4v1fglnVVZkFG4mPRAYzwb48TUWmEiGgz1+OjxkWeQ2YWfkT0AHVYdNHxR99AI3foOXqU9LpILTA4Y1L+KMxy1oXOZQZv9EfQgxlAen5Ays1ZvYO1+Aw8xqeaaNGWXdJtcw7euJSp0c10P3cDwT/qj84aJTRNRRheexlFwg9m3LAbEsnTcWQGutx2MOQXjTgh2LSbWJdEFiNoKs8y2YDIzZMsepeBB+6F9XCfvwH542zl4VvGjiSMq+KQxD7Vox9zD/ZvRYXzNTorkvSJJb1hP0tuY+bzqqLk+POzU0ek8GPKhd/iwc6tySmGEOkVwAJ2mUQC+zWbA7iZwnWcr5I1b9nT6bEV8SAvUKr/FZM+nBzhKTQXUIuiegZqR2/ZasxSWKXRCjZ6xfslWNUYkszTgt2SwRYHOPP9NibtOWEJyA2F107WGKkX2ti1jF/pQIfyUKFnQqKN0sTztgzRFY5bAYjuixTqlYbFyiNfRnnhKV9eDcaffYddD4NY/AnffwP0KxbBvD/B2WLvVgqDGcnysH/U8ni54kvCbceeIHSGLU49oumEgvSUzWFLj2U9//TfsA6Wwe1HzyANNsUoMphRhIJ1Ul9+XHgtSsT6zqgep6qGHv/qTzMiSxxU0SfY254RXM1+H/eFwOF0XivqESqyaU9/Mbm1hMc2FjwEwPcCEVMpFZUUu95NGMvW2pNla8nozLR0G7oreVGRXmLlL2elXCY9H+IPS0ikTfIWTdrvWSQuFtc2BazI4fsq6oZRH+aTpCoIKOGgC3uwEYuZkfCVv/6imrWrEVJG7nNNu4/PoGviZ9RkwM1u9AZbFw5AiuRSnlID4Ht2WUq1vyJd+hNn5zT5k9jGSiPWBSFqWbledjuRWWllQGAGZ3edIkb0xG9CBBks6rHvbSI3f23vxAbNn0dqcRt6NxWyZV06TzQGxhK1qGdyPBo8HHvtn0kPk4FbvtXwBLklP0eO4D1SotH/Camx4u/dg4SVsMWCrMod6iBzMFKX6qknMVvUDC7rBEEnJsFnqe6KB7nTgDY+esDW5U9Kl3xTChDRkWprweAWCxSSR3wIuomtPgwiTjlQv8rbemdQdNRoq6Iogss/apDyY4z5cBQoyGU96NFfoRZ30PTG3lGFsPXjkHnmzWbdtomazWfXxnjaWioz3R40JXtdPjdfXPdDyXnQ1QhEbHsF7hcfxMWCe+PMUXCIbGZitFGPfMv084FP4uap3Mqwz2e3eLIoy5EQtJCiqtQbI8WpPMfcDz8l97cBtcGrO1FZZrLJixJKiXIlNl6n9nKfCjBMfw4nOIhdjww89f44pf6HIswQcYXqaBmBVbo8+IG3HkxL06yiaB+JreGrKAgrocxiFYseVLuVh2kPjHtPH7RTW9qIDlGun5ASHTFJDSTJ82/be8XhyUWlUAkP2bIEmeRShtTXQ4HSrJjiaS4f4DVsRO5ewKm8LQKpJfZAYlCmgSP2ix1e+0iD0+wYuxF51SFLfFPhkfhaI9r4qewVxnm0jemuDzVSnFHcnFaBjPeSvlglwh7NDQrLZdjumd4EuKL4D/I2kdRfCvQC5V9RtwV5P+faWG6e/8Vk7I2xs1MBbyvtUUNJMpUAFWmhnEwoZOPOpQT7x4WNlOspaF/JIiJ8tUDE8R3ZJTeKXsXF6xihPAg8O4PY+IETiPaafgF2ajmExblUMn9KxYXbHz1Ze5IJfGmY2MrXtBjxNv/FT8F88z+zgKPAQeRfjPZ5YVk7CapTeR1emCMTS8b2xAauq0dj3hSbvAAroZvSuH7Am0t+Ha3H7s9q2fK2jzxj9U2dBKam50dv9Dz7heSrAxv/yOfdF4kU1cLrvF3RA0kxdMKfHhxaGAhzpSh8O+xYrxo/rb5MAVTgf+bQWRdzQxjTUsURjQ9tq0jl88hbLhJjEg2OZ6FFkAiBP3MTALLEfGNbOm7OMKo/QpvTYeP/6+alRHMEdHva7G+HoQZy6LuhYPwpN492Pf5uq/HxmCrCVA49j+YdE5dxgCn8XwOPeZ+ucbuoDGeKOpiR1IPAtZHolManTZ8uwB8Mtwy7yQ8SsbUrOaLDIY/av2x9llG/uEBa5X7FDeXJjvbM3tTMYjf7uBdzRJW2gi5dJFJPSuJMIE3mcwqIke8q3J7KgDSNfyLx7eoNtdhED9SV+U8IpOsBHdNOEb5zvkEbOAzTaNuGsiPotUCJUJQVQQox7yVxxgpmZ7/15IV2NvJ0NHCXp/x49MJMOdmDyFu6eS8RP5bl/Ov0mkV07XXLnv0JhSgv8foM7DSiRNhE8uMfAJCPKsX0Av0okZt8eWuzYYgP7kTzcNe7b/UExBUuRJVF6P9ze4WFZnDZds+Ieeq1ytoGwfKGMC1NuCiqsVK0LGj6lgG2obXFHLw1CwKiHeLqobz8sSFFSAj2EENY27t2PGi8ARfzoHkSgqEi2xoKqrljBhWhKFeidAS2oApJxLyZUCTjj0h4byVJjlOsuz7YdD4bsBmt5sX6/L+uZMDMBQxVMJkzEW0vT3vGfNO7ogOcSQCZUiYyZDyv9PKnMKZ4g97g6DPfzOqQx0feyKovNZNGyETtWhdcsjU0AhkN0BxsBYg09PTFwuxdVI1ZGw99Y+gyTruWUYa6BmsYJTt/55PC8e98JUzOuk+nN+Lp7B6L3tBaKg1jbZPBf5D5MCDrV3l10UrTC7f5tEPWAZA2K+0nZH2SlIzOKMVufB2CeoslFs3gvw6pyqKmBq6qwopDVNTNU03dnR/cWvLoA/h53qV0wWbG04Khy6MTH0ITcUWA3VIxRsCzykEd5ymrVkfbuPLrV4FeqeFisamXVQzVIVVXpx/8R0nY6QbhLJvRRt20zTbPFMOWMYbbcZojTTK0D8jR64RO0lyoxKl7IXIS4QPtfYFzBfEsXJMPb086qVQsp6UwmZXYno8fnG+FiFZLr7byGSfKyyGFq4olpil6k4wmZRecFtV59EtBm4ypc85ne+al0LaN0m980d9Yoe+piVon/heZf0i1dbqHbnAcSwGtYJERyoy3hj5GLJSswCwOrQWDFjoCOL5TFSO8jKbIsTzoeFmb0MXqbVLNEps2KKyXYG4EuMMlIIvs1+FuU9O+4UR5mYOBg9TuYgZLp/9Xbs0qS+S5EP73EI+KoBkd7v4IKpgOaNdvkkHbcLVZfOjBlicyUwc9cQc6o1Kj507/9L4M+Hk6kmnZ3yLc8clvDboApJA+P1vAbaPwOh6VJT59v6YOO8tbB6xqi8ne9j6HuAyuN3ocMCp8XZM3AWoFmwC527i7zJ4/YQ+Pf58spjmB4eFRQ4INY+vrgpnCpDsv4Dr+wOMa/dUU6DQTPsBzGNqFVJ6LrBMZCreAEoO2rzN7jAurX0IQMWFmRA49YbIEvz0rXoPcGjwEu/dDwMdVIL6M+avPUZ0sftUWW4Kpt/vAP3V9nKorD2k077QjxeYSaZNAn/I6O+oWVhtFI/eEu3uwW/novUlBgcgndZflUR8ZrFDxSdXkBLfXa0FxfspVyM5isfHk/4+mbCItJVM9o3ctkmjr6SN4dCpYeY2AE1jiDgjBGucp9XTkbtrmnnV3zIsKlbYO7IP8i8+hO6GvL7Rnl8gLvB0D8Oz+X5+GqYeS7vrjvirdNqNex/4NIWJ5SiBVzCQ6YTAzcwWxISsw2jOVnhK9F6gqYGX+HwPWjaqQa94bbFm2s3ZynDSw3FNrb31c26f6+zYAuIEK4XSkSOqm+o58qz8zDhORiXj3hjoeOA1lXoOIR6BrXu9pRGKlXxtOjlqWmHpjHXQ4w4NMdAvSDsiwAGK/lyf/N6mLpYK0BSdg3H999Y66Vazr9nP/494yD8ShcAatBpXzSDpbYsHpwXeJURIS2YVV48jsi1VoSYCcM26pNKUyLwzybMUXMNiDa0jlI2ppqViL/EkSGJ3OM5ID0y1FgFtZXxSF3loLilEvBTlzWqmI2M1uL766rnJGvy2F9Yz/8Xd3N/uEfd1qYdjQQLqlYvKQN4WCu6/Vyx+fxFiPt0gE3tgXQa+3hbgfTFkNVieJY5O3sO1CwMlP6rkDvUsdNiT3qefQVxbeMNq1Y8Gaba6hT89t1/adoqgwXtAJNw1D1g3QNIR7HwY0jo5syO7lRKgUGCFaCxXCB0HY+tqt72frcjm6NQbWs0pXswHYXPJwLs9qpxfTvSXsFIZDkZj0XXK8oKRyFzSLVaCk9ZknNYSlOsiQfWKXcWWoloRNw5SBMoLAsrGeWJdlapWzyG9Vp+42Ktf2qFdvaDe8JMFlJfzkbmsiI+Tmm5YdunoAh7N6UoaC91thC2ZCOiqDljwV4REIm9v4+FWeU4Bv61U5zFw9jmdUKMxYj7CaKLarY3Qcvdo+eiUdVvyW//pLOyt6UMlCsWtUmliqUBs8qpx0rjQO+nOJhQLNyqN8qT3x2NYDJHLGnAE1F8OChLQ+5muW5cP1JOcryg2mlaypmAZCnWgbJa7C0iX9Or+BzS9rVGyamcriyArs4PqqRIXO2euppGclTzD+fAAQDvJZcmKA0OSylzl2MuuTXDuljvcOsVCSC2vvlpXCpAmKy53x49f4Pr97Xy8LOQL04mP5n4o/ysNYLXyRY6UdWyaXCpa4IPE7A0PLJqKTONE8pXAjGtEgyLHYyTdbKmKaReyHKensXsiKCH2LyJe6xXIlp1WjCMInGlcruXbQUN/CwPg+e0q7X6Gkt50FAgQOiVDR8lc1H+dbOoWdNrIwsD0OB27clcngkuB2xzFYlTDONA5IiRlIktM4hpSxG9PqKHfW7jSwlSUNb/pJWfrrDsFN76mOxB6Nv038YtmqxNHWR0jbifPuBKsGsw65Vw9AQAH/FWgHPQ3fhyGmm4Y0fPX6oB0YDHa/xXyFB8K7MQC3lSgI1Kek8obJw48rQ1GPqS9JT9q6OpOIhCrFppx9j0SLMHFnYzcEkteKz6MoRSAP1QOeDDNpClhTZFp5DCevjSfPIe3UvQikMzVUTLRJU2xFFf10VkMy6xV/u8ors3GtR+3tMlSbGlhoT8Fm1tmT9iDKoqLmgFJTeZ/r5LW7QXG/JkjAWWRano4ODObBsPrXdaFnBpIrUAVgMgqciPcC83zQ7QBcdFe/BZo/bqHzfA0Ll1z2+9B4eoQalJMZGQRAYq7sA3Y2jtFg/enR8XNOu8LjOqqCH8C98paCZYRWBwebAUGOD42FEzb+DevEk3bxeuAKfrNVDzZMg5TNhnnT/KQWgyvOM59libOKorALPlu1rbOYsgQ2wHGRbhubusvKLxEMujOSYSm9leLycGk2hwdWjJjFAaLiluslrelHKSV2CquWn3PpJ/TMqZjIhBir1vdHrhVEP/3aILOghH0G3u7jaMxISkJHB8BHN22C0wqm63SVSLU/Sj6sYvj17hTYq+A9J9fmHjy+BqJUT+ZU66FiwoCxFVq4zVLO9XqQsY0/Zw36jgJQ85IY9qaP9NrJBUK9pQQYo1q0TYApiUMJMCvUw4b0v/d5J7/yr722YnnI2vkeVgYFN7AL/OIaxVt9nub4QyQlf2pgSHZstDjrYJLysNLqpdKc2+SZU5xmXZweX58o4z7dU9qwzpeKVc0WnXc6fy8KkRvlnGRunzFH5FlYGDnmtQKGC8F2qi/KLEGs4MfmHIaNKxXn6s5GJTGSNbNs2GiZw01ZqV0hrSol0jVQE7apms+ZoZf37r6UlOxBihBFq3pYpWtMbalmA1lLzl4jv/V/PjVkC", "estudio.html": "eNq1O11v20iS7/4VjDITkglFfVmOLZnKJo4nycLZZOPMzgKGkTTJlsSYZHNJSpasCNhfcC8H3N3bYR7ugHs44IB72Ld72PyT/QX3E66qu0k2JdmTmdkbICOyu7q6ur6r2D6+5zMvXyZUm+ZRONo7xh8tJPHEadCsMTqeUuKPjiOaE82bkjSjudOY5ePmYQOA+XBMIuo05gG9TliaNzSPxTmNAew68POp49N54NEmf7GCOMgDEjYzj4TU6SCOPMhDOjrN8pkfMK33/LglRvaOs3yJv4OUsXzVbLqTwf1Ot9PrHA6bzYTENIR3t+N39+E9DGI6uN8lXb93AK/5Ih/cp4f0iFJ4i2bwdkSOfELgjXje4H7f7bvjNryxK8AyPvIP3PXew5XLFs0suAniycBlqU/TJoyskSeWy/zlKiLpJIgH7eGUBpNpPui0298OXeJdTVI2i/3BnKQGUmoOPRayVL4DNeZwDGwZdPaThZYts5xGzVlgNUmShLQpBqxzOmFU+/6VlZE4a2Y0DcZDNqfpOGTXg2ng+zRe793PWbJKWAZ8ZPGAuBkLZzkdwigQkyyGIR3n4imVFMKjH2RJSJaDcUgXwwlJBocwSMJgEjcD2DkbeCAwmg5vmkHs08WgP0TI5nUKoPi/9Z6d0cmqhmbr1Fwk5lDwbdDBg7Iw8DUxiwIqJpsp8YNZJmhLiO8jv3uJIK2bLMR2mjvLcxZbtpvHK2W3PAX2JCQFkmtcBimXu7c3dnqsbATPWgd2ERIJ4ikwGjDN0gxQJSzgnLieAmOasI1HBzGreCCJslmNpPtdrzvukS2h19YMgH3EDam/YoA2yJcDu9cv9vXpmMzCHBckK+TvoAPPvl+JOqUhyYM55cOjTaZ8vQh20Oj7mh3ReFYKOGYxHW4rmdCotlC2zi7N/5U6cADPURALZzHoHOFcRBZNaW0H7fm0sggyyxkn3mYJjTdO4IbMu4JZHNSIJX6FHOowQ7kXGnJOF3mTm8UAzWjbhjmATz2WEs4ZwSdJ/CHqFVJcPyCeSeETX1Jq6R0qWBI/mOKRa0cQQzs0sFiURSQMV1vWAXwj+YCLETwJXSS36BG7KsWk4sBhpLh5XcijDWhcBiRFO3ySmBhwW+NuqY88RusdszQSdgw6Tf9oNGHGVLxPzdFwV+QHKfU4eiBoFsW7vFfh2FBjhFSPDubXQCGJgyhbbTlBlEzdzw0/zbI8GC+bMoZJzAWKQoF+td0prqpQk+6GHfyki9ogyt5FF4S6nYLk42Lg/ng8BlRBPGY7RCiiCQQtbvMHSOO2UqFGQNCkQtKFFPeHeO7CdDt2H3YJ6YTG/uo2z8LXS615vGH7+/1N21etSiYCnveLXU9pvRvHqbnE8mzlWTQ/mO/WrB0KKrMHQKu11wWCYFV4IFhUphVbbqSn0BLEnLXSx90Ho97F0yCGZA18TI26HVTt1vltOReH7633PBLPSbbpbH8TUT8gmlHZ38E+HMRc7Wk8ayk4g6p0WPiEwzJTOcTAr21FftUmjury6fEVQncRqdAKoT+7N1BdvDyngq9T4dPctBYNcVw6OrlBry2gc8bCbFX3OaWiNhcyTGl1D1Icqr9D6+SpspxMqEv8Cd06W6ff3uBaYVk1gM2TrfeOWzKnPs68NEhyDTN/pxFEmLlHJGmMVvIlawxWjXyaUtoYNKZ5nmSDVsvzY/tT5tMwmKd2TPNWnEQtDvSbtt05aNvtljsLQl+M2RHzZyGFFQ1LoGrBqVmctX4eSrogEWTJWetTFrUa6zUcghMPp2iJ6gRTc3gDU9QC32kAu6Cw0DQ+4IUky5wGaFWDT2KdkvFpABDS0HySk+YcFtJFo1jA4sboPSjLLCXHLQE32oSHJBQYNXob0BuSlVA7UYuIicA0Y9mtGGHzZWP0NPWCMCQqxuMWHGbrUHDOnRO+Lw4b4qaNchc5C1bVGP3tX/6snc1utL/90/9U1CgoMI3YxaYQ60JeqzWKom33aRBwAQYWsBQg5ZNmgHMy72AVrIIfcHwehWVPi8dbt/BTEkEu5pHG6Dk8fvkRn2+FBnOOAfIt/mhGCpVpRlItF2KuyBI8vZOzHomi3Yz933/9h3/TTr78CM6e/Gzuek6jbbUbo++wuqG7zwEwnUMOdQr1SeiTW8GOONQZ8ZkW3PzJvoPtANxUoCHu2Ldi7fWtDlT/vdb+rSBt67CNapwG7u3UjaG2aoxOiGpgP814SFp38/10gX4LxHknzwscQgC1/Yqfwo1gWOX0TUgMPAFvRkP2tz//+yZc5agbGnevQOzddRT6834Vl2ppU7+W1sg2RZWqbaeIO9LIejAZ8qbGFOR6PWhrba2DqY7EfHhYepDq3DyUbI9jWNweFRGn8rc4xkPdLa6pdMQ8bm7YAI6nLK854acQ+JowSLadADfEHPzECVhLqgBsIb2G8gHsnoVf/nPC4h0OmAvyioZoVH+a0ZDmbDfMFMgDG//Hv2gnJKlFh8pLFz+1ICuiIRxYBFjtoUYy7f3Ld6en2jhlkabzqKcPi/nVm9QN8hMQZQqMWqtARRz15GSrBgrxVMHy4uz9d2egyVAw7EIR8qmsVYHVlz9/9/TkzVesV+DqCN4xFp3G8yBlMdhcvhMJreaz1sYCgW0Pjprl2jdO5ox85s34DMgpXZ6DrLycpUZmDiXUd6/OTs+dFbj1wYUuzPbDNXXtSejqVjGAOgomUR8UkSC4KSYuLR7iAU3CY7wclMEchzGay1GM3TBERPAWg+uCpt+9Ojl1VpvUDPQXZ880iNI0ZRoMagYEMY9pj7Tfvj19Ye6mViySI5oByyztBfNZbmnved71KVNWgouxx+4CVn337I/Ym9VooeCa8X0c5EtL+z5OKQkt7XUAqRaDxXuaXG67IfgCWPwMfynEbzFi3s4zQV4WKBtVwJA6YLRmNnM/AeSbZ78FqJxHbaZs+wG0J6UZOM2eb2d5CKDn7880Ofrlv2Ot9xzqjEghI2R+50O/LUk4e/O8YGu//W0Nqvuh01egotmygOz0v+U0KLIe6CK500CZtekMpA24KqEP9NckIRAuNT4Gc6r0gWjw2xo2OwhMzQOfsg+QOAQxSe0o2QeAP+CYkKWH5yrBJqD/PUh/VTB4/+t/cQjg4ocE0xdiJ/EEAF5FEINiJIRPamJSX5eWk3Lx0dSJ6bXwOvYP1H1x9k6OGysSY2c+INkgT2fUQk7TdE5BIa8h3jybjcdQVuPUGgytQAfVWv42WNDwHTbEjNckn9pwPkM0/asZq2uaQ3XROUQnI4hjmv4gPwzA40tuVyp6NsuTGXg2CH/n2It1BOnn7148qwYrxOCGKUgkAYIl5NOT0/PvgjAKvPfVnEIJD4swbNOY92YdPOFwr/QxWFrYsAoWnEyhtDHKpT64qZAiUOl4MigpqcLhc3w3zCEft6u4roDwUxjthfiwUWJKopRGCtjb1+9OX78ALClBX1cQgU0cBAcHCrWKw1fZ6GDFzrh+w50apgU11b5pC7OlxYaglup24NIT7LfN6QmJYFMDkr5KWi1FWoiuY3Xb7ZJ2Lw85qlpQMmADayfvAFwy/znUeSg5LgIcxpr3HUT+nO4aO08oyKtjd4uNJ2mgsvYFvL6kYQLK3Wlb+5DdLrq01xEP3e5+1yskA2HIwMXlEaB0Z6qpvKbZ1FDYA/UDfUFZRPN0aXTbcHzTUsQulQpLHhIaart/bfImLEttzGgwK7QXTpObzdtXrW4xByUPsF7gEUevCOUgJaVTGgUKoS/hNUumwOQzFA4o1pj/B0fuwX/7+5Z9oJ4al1fqO4sVVM+LjisJN3B1AcMstj2S5TUSYUzYkx2RBA0cDR3Ys39o4f/UfQG23DYNoru3PTwkBLbtqAhgESDYg+iieTMQ1SwMrQjcjXwkHEXmrNYQlpl3pVocvoMZQFasjD5jix4MvgR9OrQwERSaI7Bh5ouoyJwEQrfP4WDlAUQOxMerNMoobNPHiM4nlRwJZvk4cug59SA4pW9BCQz9l/ZFwsDNWhxlaxLm45ZuDgVZfAdlYw6DrBvPYs4kjTcJjCts02mC5DNnJWv8wUUldeuia+1bvUurkghok9Wx2xbK/9Iqyn25qOv7Vs+GZX0LFACXuWP/EJbZfcs+wqViGSbrZc0v1pL2ARgVQFw092Fjvvig/biNewJGMCT+T6wvWwAKsT3Y46Jn9a1mB9eOx/3+48cWytYGojt9a0zCDNbyhoCyzt63LtqwrC9WCdOBdQd8X7FofXF1OQSiuQ2g+0YWv6QL4+yifSlMAxv0MVR9S+fsonPJh4oqkNuEbdtnF91L045mYR4k4fLcIyFJjZcQJINoE2vvUgyrWPc5CThaQ9zsPnxp4b9m7yEgQ/OuLetfShtSUm4Yfnz5RISQAer7NsgrBcUB3zkYG/dwnbnasccj+wC7jaWGuWBrhrkCQ7VnCZT/EIfzNFj8wFIIqChE7MwvwGtc0dMoyZel7bhbFop7Ix4o9+YQpKjBnNEKiGF2kKGXNouX8ysMUr4YAzu74ploLAkARAzYHEFyQZ9hSMYUB4kcujCcLAGFWw2bmAGES0H1PsxFFf3mmhebFDQDtpnIoPATuEuwr94E+TOLgZmGa67XmEtB9AbOsoXC5zFYAjXIjQO2QUOn0+U2LbgvOeo5iGlC8xPe/VaC2h94GQUsNq2sAOJZ2i6Q4UtH5HpkYWT20oIcQIhGuDx1cmEhQGbfmA879lGLz+QkxoTAHrP5wyL4QXZrQgQskRCH3JST2PCiDg3VAQ5KNgzAsxePOAwUIAYxBbzHMoOaD33Ls5fVrBy5eVTCbMKLI0HSAd4JuCGkh1/OYNeYktTxW5D48Ncxvj3sixehY29T9kmEMiFTQ+Q6pQKqFoJlUQYmgvEsd9rW3GkP71LzBw/uMXsGCfpz8Jw2djfAvEQS5FTqNZw/ciY2yWF3F5Qxq1jlgd7lwxynedvpifwVE63e4I5lrR4qYCmnr1AWgP3G0Pl3Fd20RfL4/vWZ8/GbFec3T4qNHJJSdsbwZtA5bB1PDHOtwcOXH+PJLGSZ9te/aN+s5juA5l/+I4UAQLNjNx19swKdA6DvIBvwjS5Mf/lnDQeXuwZvaoPRx0dozQZSavNyL3vyUeyrDK1FJZh9HOg6mD/JlrGnlaKkC6jAMmNsrkACUiipQ65JAHklzb2pMbZWIJ4p8wf6y9Onz/XKnKHguVp7BIFAoHKQR531em9zo2zKrg38nKFEbo/EfubwBsgFTl0OUaPGQUh5JjOEWtSQKa7GxgLcBL3imY09JUi3ueLw46GbUnK1ls4ex0xBUSFPTDBAnrznacuWp6PjZ0a9Uo+JPLrMRvDnKZ7DQIRg01ytKUghJ6GpYMX65ETeJPu43YTVaqrTabcfUo6b+q0C2frbj0JPYQOwJXMlwhqUSGxO+cgaZqpcbwNAmRhuJIRrEYTAePiK4d0BiW1lymxXhs+gcJtB1hWdwA5Q2HCxD3lwAXQoTMdxdGwy6SYPDry62ChSzsGz+pBKVcWHaB1D7nXket6BBfyaTGOaZQO737fw/l4o3tp9wazabrLRpeMhvnpHYAMIjxe2spegbPp4DeFL7FQl8yiL4WakimLuWKIgtpecw6UXWjadKJalUjUIHhNruPoIIK2qgmFVFQxR03ibuuaQdF3qC7g/7MlwcDuk8SQHQUpM5fGfFiCvcaI8RaH5vC/vXOjvKJCEfaMT0frBJ5ZCGQwP5ySc+UQ+5fzhGVgif3jLMvrhvX4pcNYoyliaGwaxXNMZGXwf4b7fgCXbeB/TfHR0ZH57dNTcmHVrs5Lcyil46BTUncyV5NmFx1deOpwJthcGyVM+g/GQYymO7Tpl88QDB5JTWekbuuiT65gHqdYtMINJgP/wqKF/gLNrHIrFsA8UbwYcE52LIUDNmvTU5owrszEOXFBeCODySfE0UI8ISbvEKo4m4zO2NTg2dKDgGM/BBUibvM3c1ZSzWCEbZkgwOhGgt7gF6NyTMOok/+5xBgGk8EI6ZxhaC81/gDBvIDDG+xowfgggQZwJ8DLgii85Wy5607o/fy5GeLdbf6LzCx36QMfvV/oWtspeNjE90Y/d8gv3CKPxCfGJxn2Q5mizWEROG2eeM9BJOJYW0Rs4SWxpIdHw2hrBQZ+5IbH1ARfnJnm4SfHNne+iPxIhW3RuP3+GuPxI18SbBdsSpX87EEYucmWoknn1CyI9E71HmYkNbw9wJVN2f4h4GoawlF8z0AqNh0Q3PSUQ1V1n5Cpyy9lkEgohW66NH0ZBzPYczsrjOs8RZYsDnY3wYlXWyNU8LjI/SJilxseXQwzZpIjXIgAihgcP5MM9xyGmfIbk1advZlCW9swhsbHxmxsmH30V46DNNzKHBRnkp06v3nX52tOrLsFx4i1/YJpqviytgcW84LuHvlceVjDjOnMuLm+x1HrefA357SybGswEO4WXgthrZ3Rti2vGhTFem7fWn/eKeP/58+YGVdLEQ8tWcWo8TVOyhFH+a1SR1nxSPQ8uqufLiqORM4r4LlydHWCHTCXX5XalflxvNjSrQsGqzzwjWeBtpxCH7qE7HlvldiK4K9exxYDS7rR8muTTH9IgpwORxppltLius0kkQde26BG/4bGzC0kR5gfXWPKuy24ft1RHly0qXenWb6ticVd6UwuL0IIJKAULZwmUawmZ8JhQ9e4cV6qADGJynDn+bu8LoUi467tJUqhZOKPFLrfPMZncjpmp7oYMkdOYR5UbwfjpHB4QCD8UGDo/oW5h9Pw7kWPeyesLcc3m8jZeIyGrUn6Vvws3nfD6a/bxfmKf+vcDEbnRqacsvzPQBpglFaR5GHbwogqkwD+7jyJBc2d70rMXlmh5LSCdffnQbj/GVkSRP252NXIAz+0l/Lt5BMD9vmglqG2JfKO7ULkBQcUFsSB7U05mQzgLICuzgBsRSYzfzSIXax3ZRLLoT4nhzhinyoIXqkp4QzWqJHGX1O7V33fKT40idXBzLfbx8nRjnyL47ynZVJ0OYeaxc+8r8q3hThiVLvTLSsiSdMmc746Nt7LCHRtvwWxuvFfGAbWMrb5Wyc6o/FTGq5gKsAw6Nnfl78HpSlvaBYPXksVffXXVjz1KJS30+856+VfGw1VkKyFJhJXIlkHJ4d/geDSRPduvrfmH/99EFxR2hvUDyLiJNBd6M902nPLTqngw+LEscCSFFyK3lmVEx6xvCkHd2fGBFtQJY/T3784MPcBbBi28cYArfHYdY4bs6J64CcXvIsAEJ8tALd+OSXh/44aKoLRCP0f4h2Zn98flu7upX3ufYPPCQlm3nzGWGIrZ+VCMYuWDfv05hXrckEXhgwe12tDfdra3cR/33jN4545vBGqHbUH+RVE0xgy1NainweQDL2U+ZbyU47/Ay7IpWHl0KJXKb5FvXGSOPSch1DiiFiKpNw3mDKqhlcy6Npp/v/JulKXek1EvxdQvwVyaQKf4cL2LCEn4FV1mBt6RMjfABXNA+FGQgTeBuHNh2/h7yYOWbIkKaxSwZf/VFE1NgUuy4JeGs6poF71VJZpd2hkk0QbQUGuiKh3yyEGrlZdO9a/jwlZP9mdY8Vg1TnypCnZEfjG+fPRRO+Z/Ijb6ZjUuEgEbDp2AQZh4w55PfgRXpLZXCHebG7RzMpHeexL5r6NVLQPHO/cX2H/PdR/c0jlFTX9LIKRnBlgvkVkTjnJeY9Uwd36PRm3oc938/FnPsUWNJMtO+W0ynDuqZDgwKEYQ+8aVMxJrr25bC9vwZq2Mv3J/kSZgVtnRzZ15g8jvZTIqr/fkBfk0JwlyD3vmecFol+tXdQ15q8NGcxjY6GKIFs+6NDHxNQE7DiYmRuUfWRy3+F9XHLfE34f/H/mQjk4="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()